# 12 · Controles solicitados durante revisión

Este cuaderno añade controles **post hoc** sin
redefinir los estimandos primarios del manuscrito. Responde a cinco preguntas:

1. ¿La diferencia InterGATE − $q$ supera la esperable bajo nulos candidatos
   emparejados?
2. ¿Qué ocurre al completar el factorial con `InterGATE_raw`, eliminando la
   heurística modular tanto por semilla como en el consenso?
3. ¿Cómo cambia el tamaño del núcleo al variar $k/R$ y el número de semillas?
4. ¿Por qué los folds del consenso y del núcleo estricto no son intercambiables?
5. ¿Un reentrenamiento con etiquetas permutadas fabrica un núcleo reproducible?

Los bloques 1–4 reutilizan artefactos ya calculados. El bloque 5 ejecuta por
puede ejecutar 49 permutaciones con reentrenamiento completo cuando se activa y permite reanudar. La concordancia
externa se lee exclusivamente desde resultados congelados; no se abren matrices
ni etiquetas externas.

Ningún bloque se usa para seleccionar la arquitectura. Todos los tamaños estructurales se derivan directamente de la ejecución.


In [ ]:
from __future__ import annotations

from collections import Counter
from hashlib import sha256
from itertools import combinations
from pathlib import Path
import json
import math
import os
import re
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/tmp/intergate-mpl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    display = print

    def Markdown(value):
        return value


HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (HERE, *HERE.parents) if (path / "intergate").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encuentra intergate/. Abra el notebook dentro del repositorio."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.biology import (
    DifferentialEvidenceConfig,
    combine_biology_scores,
    compute_differential_edge_evidence,
    module_aware_exact_k,
)
from intergate.biology_validation import load_gmt

try:
    from intergate.config import CFG
except ModuleNotFoundError as exc:
    if exc.name != "torch":
        raise
    CFG = None
    warnings.warn(
        "PyTorch no está instalado: se ejecutarán los controles post hoc, "
        "pero el bloque opcional de reentrenamiento no estará disponible."
    )


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})


## Decisión analítica

El bootstrap de aristas no representa incertidumbre de reentrenamiento ni
independencia biológica entre aristas. Se etiqueta explícitamente como
sensibilidad descriptiva. La prueba basada en nulos candidatos es la
inferencia comparativa principal de este cuaderno.


## Configuración de rutas y cómputo

Las variables de entorno permiten ejecutar el notebook tanto en el árbol del
manuscrito como en el repositorio autocontenido. Revise esta celda antes de
ejecutar todo.


In [ ]:
def _first_existing(candidates, *, kind="ruta"):
    candidates = [Path(value).expanduser().resolve() for value in candidates if value]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"No se encontró {kind}. Candidatas:\n- "
        + "\n- ".join(map(str, candidates))
    )


explicit_results = os.environ.get("INTERGATE_RESULTS_ROOT")
RESULTS_ROOT = _first_existing(
    [
        explicit_results,
        PROJECT_ROOT / "results_v8",
        PROJECT_ROOT.parent / "InterGATE_manuscript_revised_20260821" / "locked_results_v8",
    ],
    kind="la raíz results_v8",
)

explicit_resources = os.environ.get("INTERGATE_BIOLOGY_RESOURCES")
resource_candidates = [
    Path(value).expanduser().resolve()
    for value in (
        explicit_resources,
        PROJECT_ROOT / "resources_biological",
        RESULTS_ROOT.parent / "resources_biological",
    )
    if value
]
RESOURCE_DIR = next((
    path for path in resource_candidates
    if (path / "PAM50_symbols.txt").is_file()
    and any((path / "msigdb").glob("c2.cgp.v*.symbols.gmt"))
), None)
if RESOURCE_DIR is None:
    raise FileNotFoundError(
        "No se encontró un resources_biological completo. Candidatas:\n- "
        + "\n- ".join(map(str, resource_candidates))
    )

DEFAULT_ARTIFACTS_ROOT = Path(
    os.environ.get(
        "INTERGATE_ARTIFACTS_ROOT",
        PROJECT_ROOT / "artifacts" / "intrinsic4",
    )
).expanduser().resolve()
explicit_dense = os.environ.get("INTERGATE_DENSE_ROOT")
DENSE_ROOT = _first_existing(
    [
        explicit_dense,
        (Path(CFG.ARTIFACTS_ROOT) if CFG is not None else DEFAULT_ARTIFACTS_ROOT)
        / "INTERGATE_V8",
        RESULTS_ROOT.parent / "dense_seeds" / "INTERGATE_V8",
        PROJECT_ROOT / "artifacts" / "intrinsic4" / "INTERGATE_V8",
    ],
    kind="los cinco transport_support_dense.npz",
)

MANUSCRIPT_ROOT = Path(
    os.environ.get("INTERGATE_MANUSCRIPT_ROOT", RESULTS_ROOT.parent)
).expanduser().resolve()
OUTPUT_DIR = RESULTS_ROOT / "12_reviewer_controls"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL = json.loads(
    (PROJECT_ROOT / "configs" / "protocol_v8.json").read_text(encoding="utf-8")
)
LOCKED_SEEDS = tuple(map(int, PROTOCOL["outer_seeds"]))
TARGET_CLASS = "Basal-like"

# Compatibilidad explícita con locks anteriores a la incorporación de
# `definitive_locked_score_terms`. Los valores definitivos no se infieren
# de `training_score_terms`, cuyo peso de soporte histórico era 0.25.
BIOLOGY_GRAPH = PROTOCOL.get("biology_graph", {})
CANONICAL_DEFINITIVE_TERMS = {
    "absolute_centered_class_delta": 1.0,
    "signed_context_agreement": 0.35,
    "cohort_support": 0.075,
}
protocol_definitive_terms = BIOLOGY_GRAPH.get(
    "definitive_locked_score_terms"
)
if protocol_definitive_terms is None:
    warnings.warn(
        "El protocol_v8.json usa el esquema anterior y no contiene "
        "biology_graph.definitive_locked_score_terms. Se aplican los "
        "valores definitivos bloqueados: differential=1.0, "
        "context=0.35 y support=0.075."
    )
else:
    for term, expected in CANONICAL_DEFINITIVE_TERMS.items():
        observed = float(protocol_definitive_terms[term])
        if not math.isclose(observed, expected, rel_tol=0.0, abs_tol=1e-12):
            raise RuntimeError(
                f"Lock incompatible para {term}: {observed} != {expected}."
            )

LOCKED_CONTEXT_WEIGHT = CANONICAL_DEFINITIVE_TERMS[
    "signed_context_agreement"
]
LOCKED_SUPPORT_WEIGHT = CANONICAL_DEFINITIVE_TERMS["cohort_support"]
MODULE_POOL_MULTIPLIER = int(BIOLOGY_GRAPH.get("module_pool_multiplier", 6))
MODULE_TARGET_COMPONENTS = int(BIOLOGY_GRAPH.get("target_components", 12))
MODULE_MIN_EDGES = int(
    BIOLOGY_GRAPH.get("minimum_edges_per_seed_module", 3)
)
PREFLIGHT = json.loads((RESULTS_ROOT / "01_preflight" / "preflight_summary.json").read_text(encoding="utf-8"))
EXPECTED_K = int(round(float(PROTOCOL["kappa"]) * int(PREFLIGHT["n_candidate_edges"])))
if not PROTOCOL.get("validation", {}).get("full_label_permutation_retraining", False):
    raise RuntimeError("protocol_v8.json no preespecifica el control de permutaciones.")

N_MATCHED_DRAWS = int(os.environ.get("INTERGATE_REVIEW_NULL_DRAWS", "1000"))
N_EDGE_BOOTSTRAP = int(os.environ.get("INTERGATE_REVIEW_EDGE_BOOTSTRAP", "3000"))
RANDOM_SEED = int(os.environ.get("INTERGATE_REVIEW_SEED", "20260823"))

RUN_LABEL_PERMUTATIONS = os.environ.get(
    "INTERGATE_RUN_LABEL_PERMUTATIONS", "1"
) == "1"
N_LABEL_PERMUTATIONS = int(os.environ.get("INTERGATE_N_LABEL_PERMUTATIONS", "49"))
PERMUTATION_START = int(os.environ.get("INTERGATE_PERM_START", "0"))
PERMUTATION_STOP = int(
    os.environ.get("INTERGATE_PERM_STOP", str(N_LABEL_PERMUTATIONS))
)
PERMUTATION_MODEL_SEEDS = (1234, 2025)
N_PERM_MATCHED_DRAWS = int(os.environ.get("INTERGATE_PERM_NULL_DRAWS", "1000"))
ALLOW_UNDERPOWERED_PERMUTATIONS = os.environ.get(
    "INTERGATE_ALLOW_UNDERPOWERED_PERMUTATIONS", "0"
) == "1"

if RUN_LABEL_PERMUTATIONS and CFG is None:
    raise ImportError(
        "El reentrenamiento por permutación requiere el entorno completo "
        "InterGATE con PyTorch. Active environment.yml y vuelva a ejecutar."
    )

if RUN_LABEL_PERMUTATIONS and N_LABEL_PERMUTATIONS < 20:
    if not ALLOW_UNDERPOWERED_PERMUTATIONS:
        raise ValueError(
            "Use al menos 20 permutaciones: con B=10, el menor P empírico "
            "posible es 1/11=0.091. Se recomiendan 49 o 99."
        )

configuration = pd.Series({
    "project_root": PROJECT_ROOT,
    "results_root": RESULTS_ROOT,
    "dense_root": DENSE_ROOT,
    "resource_dir": RESOURCE_DIR,
    "output_dir": OUTPUT_DIR,
    "locked_seeds": LOCKED_SEEDS,
    "matched_null_draws": N_MATCHED_DRAWS,
    "edge_bootstrap_draws": N_EDGE_BOOTSTRAP,
    "run_label_permutations": RUN_LABEL_PERMUTATIONS,
    "label_permutations_total": N_LABEL_PERMUTATIONS,
    "permutation_batch": f"[{PERMUTATION_START}, {PERMUTATION_STOP})",
}, name="value")
display(configuration.to_frame())


## 1. Reconstrucción del factorial 2×2

`InterGATE_module` reproduce exactamente el grafo bloqueado. `InterGATE_raw`
usa el mismo score $B_{c,r,e}$ y el mismo Exact-$K$, pero selecciona por rango
sin las bonificaciones/penalizaciones modulares. En el consenso multisemilla
también se elimina el pase modular; así la celda “raw” lo es en ambos niveles.


In [ ]:
def safe_name(value):
    return "".join(
        char if char.isalnum() or char in "-_" else "_"
        for char in str(value)
    )


def _rank_scale(values):
    values = np.asarray(values, dtype=float)
    if values.size < 2 or np.nanmax(values) <= np.nanmin(values):
        return np.zeros(values.size, dtype=float)
    ranks = pd.Series(values).rank(method="average").to_numpy(dtype=float)
    return 2.0 * (ranks - ranks.min()) / (ranks.max() - ranks.min()) - 1.0


def _sha256_file(path):
    digest = sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


CANDIDATE_FILE = (
    RESULTS_ROOT / "05_biology_validation" / "candidate_prior_for_null.csv.gz"
)
if not CANDIDATE_FILE.is_file():
    raise FileNotFoundError(
        f"Falta {CANDIDATE_FILE}. Ejecute antes el notebook 05."
    )
candidates = pd.read_csv(CANDIDATE_FILE)
required_candidate_columns = {
    "candidate_edge_id", "source_idx", "target_idx", "source_gene",
    "target_gene", "prior_weight", "edge_type", "null_stratum",
}
missing = required_candidate_columns - set(candidates.columns)
if missing:
    raise ValueError(f"El prior candidato carece de {sorted(missing)}")
candidates["candidate_edge_id"] = candidates.candidate_edge_id.astype(int)
candidates["source_gene"] = candidates.source_gene.astype(str).str.upper()
candidates["target_gene"] = candidates.target_gene.astype(str).str.upper()
candidates["null_stratum"] = candidates.null_stratum.astype(str)
candidates = candidates.sort_values("candidate_edge_id").reset_index(drop=True)
candidate_lookup = candidates.set_index("candidate_edge_id", drop=False)


# Reutiliza exactamente las firmas ya fijadas por el notebook 05. No se
# crea aquí una segunda familia de anotación con términos distintos.
from intergate.posthoc_biology_common import load_locked_signature_resources

SIGNATURE_SOURCE_MANIFEST = (
    RESULTS_ROOT / "05_biology_validation" / "locked_signature_manifest.json"
)
directional_signatures, _, _, signature_source = load_locked_signature_resources(
    results_root=RESULTS_ROOT,
    resource_dir=RESOURCE_DIR,
)
release = str(signature_source["msigdb_release"])
basal_signature = directional_signatures[TARGET_CLASS]
candidate_genes = set(candidates.source_gene) | set(candidates.target_gene)
if not basal_signature or not (basal_signature & candidate_genes):
    raise RuntimeError("La firma Basal-like bloqueada no intersecta el prior.")

c2_file = RESOURCE_DIR / "msigdb" / f"c2.cgp.v{release}.symbols.gmt"
pam50_file = RESOURCE_DIR / "PAM50_symbols.txt"
manifest = {
    "msigdb_release": release,
    "primary_excludes_pam50": bool(signature_source["primary_excludes_pam50"]),
    "primary_terms": signature_source["terms"],
    "sizes_primary_no_pam50": signature_source["sizes_primary"],
    "basal_primary_genes_in_candidate_prior": len(basal_signature & candidate_genes),
    "candidate_gene_universe": len(candidate_genes),
    "source_notebook": "12_controles_revision_q_permutaciones",
    "reuses_notebook05_signature_lock": True,
    "external_opened": False,
}
DIRECTIONAL_MANIFEST_FILE = OUTPUT_DIR / "directional_signature_manifest.json"
DIRECTIONAL_MANIFEST_FILE.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)


max_candidate_id = int(candidates.candidate_edge_id.max())
full_edge_index = np.full((2, max_candidate_id + 1), -1, dtype=np.int64)
candidate_ids_all = candidates.candidate_edge_id.to_numpy(dtype=np.int64)
full_edge_index[:, candidate_ids_all] = candidates[
    ["source_idx", "target_idx"]
].to_numpy(dtype=np.int64).T


def dense_path_for_seed(seed, root=DENSE_ROOT):
    path = Path(root) / f"seed_{int(seed)}" / "transport_support_dense.npz"
    if not path.is_file():
        raise FileNotFoundError(
            f"Falta {path}. Copie los artefactos densos del notebook 02 o "
            "defina INTERGATE_DENSE_ROOT."
        )
    return path


def rebuild_seed_graph(dense_path, *, seed, selector):
    """Reconstruye Basal-like con pesos definitivos 1/.35/.075."""
    selector = str(selector).lower()
    if selector not in {"raw", "module"}:
        raise ValueError("selector debe ser raw o module")
    with np.load(dense_path, allow_pickle=True) as dense:
        class_names = list(map(str, dense["class_names"].tolist()))
        class_id = class_names.index(TARGET_CLASS)
        unit_id = np.asarray(dense["structural_unit_id"], dtype=np.int64)
        if unit_id.size != full_edge_index.shape[1]:
            raise RuntimeError(
                "El artefacto denso no corresponde al prior candidato: "
                f"E_dense={unit_id.size}, E_prior={full_edge_index.shape[1]}."
            )
        valid = np.flatnonzero(unit_id >= 0)
        n_units = int(unit_id[valid].max()) + 1
        first_ids = np.full(n_units, -1, dtype=np.int64)
        first_ids[unit_id[valid]] = valid
        if np.any(first_ids < 0):
            raise RuntimeError("Hay unidades estructurales sin arista representante.")
        unit_sizes = np.bincount(unit_id[valid], minlength=n_units)
        if np.unique(unit_sizes).size != 1:
            raise RuntimeError(
                "Este notebook requiere anchura estructural homogénea; "
                f"observada={np.unique(unit_sizes).tolist()}"
            )
        saved_selected = np.asarray(dense["biology_selected"], dtype=bool)
        selected_units = np.unique(
            unit_id[saved_selected[class_id] & (unit_id >= 0)]
        )
        k_units = int(len(selected_units))
        scores = combine_biology_scores(
            np.asarray(dense["differential_logits"], dtype=float),
            np.asarray(dense["context_score"], dtype=float),
            np.asarray(dense["selection_frequency"], dtype=float),
            context_weight=LOCKED_CONTEXT_WEIGHT,
            support_weight=LOCKED_SUPPORT_WEIGHT,
        )
        unit_score = scores[class_id, first_ids]
        if selector == "raw":
            chosen_units = np.lexsort(
                (np.arange(n_units, dtype=np.int64), -unit_score)
            )[:k_units]
        else:
            chosen_units = module_aware_exact_k(
                full_edge_index[:, first_ids],
                unit_score,
                k_edges=k_units,
                pool_multiplier=MODULE_POOL_MULTIPLIER,
                target_components=MODULE_TARGET_COMPONENTS,
                min_component_edges=MODULE_MIN_EDGES,
            )
        chosen_unit_ids = set(map(int, chosen_units))
        selected_ids = valid[np.isin(unit_id[valid], list(chosen_unit_ids))]
        selected_ids = selected_ids[
            np.lexsort((selected_ids, -scores[class_id, selected_ids]))
        ]
        delta = np.asarray(dense["differential_logits"], dtype=float)
        context = np.asarray(dense["context_score"], dtype=float)
        support = np.asarray(dense["selection_frequency"], dtype=float)

    table = candidate_lookup.loc[selected_ids].reset_index(drop=True).copy()
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    table.insert(1, "seed", int(seed))
    table.insert(2, "class_name", TARGET_CLASS)
    table.insert(3, "selector", selector)
    table["differential_logit"] = delta[class_id, selected_ids]
    table["context_score"] = context[class_id, selected_ids]
    table["selection_frequency"] = support[class_id, selected_ids]
    table["biology_score"] = scores[class_id, selected_ids]
    if len(table) != EXPECTED_K or table.candidate_edge_id.nunique() != EXPECTED_K:
        raise RuntimeError(
            f"{seed}/{selector}: Exact-K={len(table)}, esperado={EXPECTED_K}."
        )
    return table


def aggregate_seed_graphs(seed_tables, *, selector):
    """Consenso Exact-K; `raw` elimina también el pase modular exterior."""
    stacked = pd.concat(seed_tables, ignore_index=True)
    if any(len(frame) != EXPECTED_K for frame in seed_tables):
        raise RuntimeError("Presupuesto desigual entre semillas.")
    grouped = (
        stacked.groupby(["source_gene", "target_gene"], as_index=False)
        .agg(
            selection_count=("seed", "nunique"),
            median_biology_score=("biology_score", "median"),
            mean_context_score=("context_score", "mean"),
            mean_selection_frequency=("selection_frequency", "mean"),
            candidate_edge_id=("candidate_edge_id", "min"),
            source_idx=("source_idx", "first"),
            target_idx=("target_idx", "first"),
            prior_weight=("prior_weight", "median"),
            edge_type=("edge_type", "first"),
        )
    )
    grouped["selection_frequency_seed"] = (
        grouped.selection_count / float(len(seed_tables))
    )
    frequency_rank = (
        2.0
        * grouped.selection_frequency_seed.rank(method="average", pct=True)
        .to_numpy(dtype=float)
        - 1.0
    )
    score_rank = _rank_scale(grouped.median_biology_score.to_numpy(dtype=float))
    grouped["aggregate_biology_score"] = 2.0 * frequency_rank + score_rank

    if selector == "module":
        genes = sorted(set(grouped.source_gene) | set(grouped.target_gene))
        gene_id = {gene: index for index, gene in enumerate(genes)}
        local_edge_index = np.vstack([
            grouped.source_gene.map(gene_id).to_numpy(dtype=np.int64),
            grouped.target_gene.map(gene_id).to_numpy(dtype=np.int64),
        ])
        selected_local = module_aware_exact_k(
            local_edge_index,
            grouped.aggregate_biology_score.to_numpy(dtype=float),
            k_edges=EXPECTED_K,
            pool_multiplier=MODULE_POOL_MULTIPLIER,
            target_components=MODULE_TARGET_COMPONENTS,
            min_component_edges=MODULE_MIN_EDGES,
        )
    elif selector == "raw":
        selected_local = np.lexsort((
            grouped.candidate_edge_id.to_numpy(dtype=np.int64),
            -grouped.median_biology_score.to_numpy(dtype=float),
            -grouped.selection_count.to_numpy(dtype=int),
            -grouped.aggregate_biology_score.to_numpy(dtype=float),
        ))[:EXPECTED_K]
    else:
        raise ValueError(selector)

    consensus = grouped.iloc[selected_local].copy()
    consensus = consensus.sort_values(
        ["aggregate_biology_score", "selection_count", "candidate_edge_id"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    consensus.insert(0, "consensus_rank", np.arange(1, len(consensus) + 1))
    consensus.insert(0, "class_name", TARGET_CLASS)
    consensus.insert(1, "selector", selector)
    if len(consensus) != EXPECTED_K:
        raise RuntimeError("El consenso no conserva Exact-K.")
    return consensus


seed_graphs = {"raw": {}, "module": {}}
audit_rows = []
seed_output = OUTPUT_DIR / "factorial_seed_graphs"
seed_output.mkdir(parents=True, exist_ok=True)
for selector in ("raw", "module"):
    for seed in LOCKED_SEEDS:
        frame = rebuild_seed_graph(
            dense_path_for_seed(seed), seed=seed, selector=selector
        )
        seed_graphs[selector][seed] = frame
        frame.to_csv(
            seed_output / f"InterGATE_{selector}_seed_{seed}_{safe_name(TARGET_CLASS)}.csv",
            index=False,
        )
        locked_file = (
            RESULTS_ROOT / "04_controls" / "locked_biology_score_w0075"
            / "seed_tables" / f"seed_{seed}"
            / f"biology_edges_{safe_name(TARGET_CLASS)}.csv"
        )
        locked_ids = set(pd.read_csv(locked_file).candidate_edge_id.astype(int))
        observed_ids = set(frame.candidate_edge_id.astype(int))
        union = locked_ids | observed_ids
        audit_rows.append({
            "selector": selector,
            "seed": seed,
            "n_edges": len(observed_ids),
            "jaccard_with_locked_module_graph": (
                len(locked_ids & observed_ids) / len(union)
            ),
        })

intergate_raw = aggregate_seed_graphs(
    list(seed_graphs["raw"].values()), selector="raw"
)
intergate_module = aggregate_seed_graphs(
    list(seed_graphs["module"].values()), selector="module"
)
locked_consensus_file = (
    RESULTS_ROOT / "04_controls" / "locked_biology_score_w0075"
    / "biology_consensus" / f"biology_consensus_{safe_name(TARGET_CLASS)}.csv"
)
locked_consensus = pd.read_csv(locked_consensus_file)
locked_consensus_ids = set(locked_consensus.candidate_edge_id.astype(int))
rebuilt_module_ids = set(intergate_module.candidate_edge_id.astype(int))
consensus_jaccard = len(locked_consensus_ids & rebuilt_module_ids) / len(
    locked_consensus_ids | rebuilt_module_ids
)
if consensus_jaccard < 1.0 - 1e-12:
    raise RuntimeError(
        "La reconstrucción modular no reproduce el consenso bloqueado: "
        f"Jaccard={consensus_jaccard:.6f}."
    )

# q se reconstruye directamente desde el context_score development-only
# congelado dentro de cada exportación densa del NB02. Este score precede
# al aprendizaje y debe ser idéntico en las cinco semillas.
q_context_reference = None
q_class_id = None
for seed in LOCKED_SEEDS:
    with np.load(dense_path_for_seed(seed), allow_pickle=True) as dense:
        class_names = list(map(str, dense["class_names"].tolist()))
        class_id = class_names.index(TARGET_CLASS)
        current = np.asarray(dense["context_score"][class_id], dtype=float)
    if current.size <= max_candidate_id:
        raise RuntimeError(
            f"context_score de seed={seed} no cubre el prior candidato."
        )
    if q_context_reference is None:
        q_context_reference = current
        q_class_id = class_id
    elif not np.allclose(
        current[candidate_ids_all],
        q_context_reference[candidate_ids_all],
        rtol=0.0,
        atol=1e-7,
    ):
        raise RuntimeError(
            "El context_score no es idéntico entre semillas; q no puede "
            "tratarse como comparador development-only congelado."
        )


def build_q_graph(selector):
    selector = str(selector).lower()
    if selector not in {"raw", "module"}:
        raise ValueError("selector debe ser raw o module")
    scores = np.abs(q_context_reference[candidate_ids_all])
    if selector == "raw":
        chosen = np.lexsort((candidate_ids_all, -scores))[:EXPECTED_K]
    else:
        chosen = module_aware_exact_k(
            candidates[["source_idx", "target_idx"]].to_numpy(
                dtype=np.int64
            ).T,
            scores,
            k_edges=EXPECTED_K,
            pool_multiplier=MODULE_POOL_MULTIPLIER,
            target_components=MODULE_TARGET_COMPONENTS,
            min_component_edges=MODULE_MIN_EDGES,
        )
    graph = candidates.iloc[np.asarray(chosen, dtype=np.int64)].copy()
    graph["class_id"] = int(q_class_id)
    graph["class_name"] = TARGET_CLASS
    graph["context_score"] = q_context_reference[
        graph.candidate_edge_id.to_numpy(dtype=np.int64)
    ]
    graph["selection_score"] = np.abs(graph.context_score)
    graph["selector"] = selector
    graph = graph.sort_values(
        ["selection_score", "candidate_edge_id"],
        ascending=[False, True],
    ).reset_index(drop=True)
    graph.insert(0, "rank", np.arange(1, len(graph) + 1))
    if len(graph) != EXPECTED_K or graph.candidate_edge_id.nunique() != EXPECTED_K:
        raise RuntimeError("El comparador q no conserva Exact-K.")
    return graph


q_raw = build_q_graph("raw")
q_module = build_q_graph("module")
q_root = OUTPUT_DIR / "q_comparator_graphs"
q_root.mkdir(parents=True, exist_ok=True)
q_raw.to_csv(q_root / f"q_raw_{safe_name(TARGET_CLASS)}.csv", index=False)
q_module.to_csv(
    q_root / f"q_module_{safe_name(TARGET_CLASS)}.csv", index=False
)

factorial_arms = {
    "InterGATE_raw": intergate_raw,
    "InterGATE_module": intergate_module,
    "q_raw": q_raw,
    "q_module": q_module,
}
for name, frame in factorial_arms.items():
    frame.to_csv(OUTPUT_DIR / f"factorial_{name}_{safe_name(TARGET_CLASS)}.csv", index=False)

factorial_audit = pd.DataFrame(audit_rows)
factorial_audit["module_consensus_jaccard_with_locked"] = consensus_jaccard
factorial_audit.to_csv(OUTPUT_DIR / "factorial_reconstruction_audit.csv", index=False)
display(factorial_audit)
print("Consenso modular reproducido con Jaccard =", consensus_jaccard)


## 2. Inferencia directa InterGATE − $q$

Se reportan dos objetos distintos:

- **Nulo candidato emparejado (principal aquí):** compara el contraste
  observado con contrastes de grafos aleatorios de igual tamaño y composición
  de tipo, signo y grados dirigidos.
- **Bootstrap de aristas (sensibilidad):** proporciona un intervalo descriptivo
  para la diferencia observada. No cuantifica variación entre reentrenamientos.

Los sorteos de las cuatro celdas comparten semilla y número de draw. Como sus
composiciones por estrato no son idénticas, los paneles aleatorios no pueden ser
literalmente el mismo conjunto de aristas.


In [ ]:
STATISTICS = (
    "at_least_one_endpoint",
    "both_endpoints",
    "unique_node_fraction",
)


max_gene_idx = int(
    max(candidates.source_idx.max(), candidates.target_idx.max())
)
gene_hit = np.zeros(max_gene_idx + 1, dtype=bool)
gene_names_by_idx = np.full(max_gene_idx + 1, "", dtype=object)
for row in candidates[["source_idx", "source_gene"]].drop_duplicates().itertuples(index=False):
    gene_names_by_idx[int(row.source_idx)] = str(row.source_gene).upper()
for row in candidates[["target_idx", "target_gene"]].drop_duplicates().itertuples(index=False):
    gene_names_by_idx[int(row.target_idx)] = str(row.target_gene).upper()
gene_hit = np.isin(gene_names_by_idx, list(basal_signature))

source_by_id = np.full(max_candidate_id + 1, -1, dtype=np.int64)
target_by_id = np.full(max_candidate_id + 1, -1, dtype=np.int64)
source_by_id[candidate_ids_all] = candidates.source_idx.to_numpy(dtype=np.int64)
target_by_id[candidate_ids_all] = candidates.target_idx.to_numpy(dtype=np.int64)


def signature_vector(candidate_ids):
    candidate_ids = np.asarray(candidate_ids, dtype=np.int64)
    if candidate_ids.size == 0:
        return np.full(3, np.nan, dtype=float)
    source = source_by_id[candidate_ids]
    target = target_by_id[candidate_ids]
    source_hit = gene_hit[source]
    target_hit = gene_hit[target]
    nodes = np.unique(np.concatenate([source, target]))
    return np.asarray([
        np.mean(source_hit | target_hit),
        np.mean(source_hit & target_hit),
        np.mean(gene_hit[nodes]),
    ], dtype=float)


def matched_null_matrix(selected_ids, *, n_draws, seed):
    selected_ids = np.asarray(sorted(set(map(int, selected_ids))), dtype=np.int64)
    selected_strata = candidate_lookup.loc[selected_ids, "null_stratum"]
    required = selected_strata.value_counts(sort=False)
    selected_set = set(map(int, selected_ids))
    pools = {}
    for stratum, block in candidates.groupby("null_stratum", sort=False):
        pool = block.candidate_edge_id.to_numpy(dtype=np.int64)
        pool = pool[~np.isin(pool, list(selected_set))]
        pools[str(stratum)] = pool
    matrix = np.empty((int(n_draws), len(STATISTICS)), dtype=float)
    for draw in range(int(n_draws)):
        sampled = []
        for stratum in sorted(required.index.astype(str)):
            count = int(required.loc[stratum])
            pool = pools[str(stratum)]
            if len(pool) < count:
                raise RuntimeError(
                    f"Estrato {stratum}: pool={len(pool)} < requerido={count}."
                )
            # Semilla común por (draw, estrato): dos brazos con el mismo
            # estrato consumen exactamente el mismo flujo pseudoaleatorio.
            digest = sha256(
                f"{int(seed)}|{draw}|{stratum}".encode("utf-8")
            ).digest()
            stratum_seed = int.from_bytes(digest[:8], "little", signed=False)
            rng = np.random.default_rng(stratum_seed)
            sampled.append(rng.choice(pool, size=count, replace=False))
        draw_ids = np.concatenate(sampled).astype(np.int64)
        matrix[draw] = signature_vector(draw_ids)
    return matrix


observed_matrix = {}
null_matrix = {}
summary_rows = []
distribution_rows = []
for arm_index, (arm, graph) in enumerate(factorial_arms.items()):
    edge_ids = graph.candidate_edge_id.to_numpy(dtype=np.int64)
    observed = signature_vector(edge_ids)
    null = matched_null_matrix(
        edge_ids, n_draws=N_MATCHED_DRAWS, seed=RANDOM_SEED
    )
    observed_matrix[arm] = observed
    null_matrix[arm] = null
    for statistic_index, statistic in enumerate(STATISTICS):
        null_values = null[:, statistic_index]
        summary_rows.append({
            "arm": arm,
            "statistic": statistic,
            "observed": observed[statistic_index],
            "null_mean": float(np.mean(null_values)),
            "null_q025": float(np.quantile(null_values, 0.025)),
            "null_q975": float(np.quantile(null_values, 0.975)),
            "fold_over_null": observed[statistic_index] / np.mean(null_values),
            "empirical_p_upper": (
                1 + np.sum(null_values >= observed[statistic_index])
            ) / (1 + len(null_values)),
            "n_draws": len(null_values),
        })
        distribution_rows.extend({
            "arm": arm,
            "draw": draw,
            "statistic": statistic,
            "value": value,
        } for draw, value in enumerate(null_values))

factorial_signature = pd.DataFrame(summary_rows)
factorial_signature.to_csv(
    OUTPUT_DIR / "factorial_directional_signature_tests.csv", index=False
)
pd.DataFrame(distribution_rows).to_csv(
    OUTPUT_DIR / "factorial_directional_signature_null_distribution.csv.gz",
    index=False,
    compression="gzip",
)


contrast_definitions = {
    "InterGATE_module_minus_q_module": {
        "InterGATE_module": 1.0, "q_module": -1.0,
    },
    "InterGATE_raw_minus_q_raw": {
        "InterGATE_raw": 1.0, "q_raw": -1.0,
    },
    "module_minus_raw_within_InterGATE": {
        "InterGATE_module": 1.0, "InterGATE_raw": -1.0,
    },
    "module_minus_raw_within_q": {
        "q_module": 1.0, "q_raw": -1.0,
    },
    "learning_by_selector_interaction": {
        "InterGATE_module": 1.0, "InterGATE_raw": -1.0,
        "q_module": -1.0, "q_raw": 1.0,
    },
}


def bootstrap_arm(graph, *, n_draws, seed):
    ids = graph.candidate_edge_id.to_numpy(dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    out = np.empty((int(n_draws), len(STATISTICS)), dtype=float)
    for draw in range(int(n_draws)):
        out[draw] = signature_vector(
            ids[rng.integers(0, len(ids), size=len(ids))]
        )
    return out


bootstrap_matrix = {
    arm: bootstrap_arm(
        graph,
        n_draws=N_EDGE_BOOTSTRAP,
        seed=RANDOM_SEED + 10_000 + index,
    )
    for index, (arm, graph) in enumerate(factorial_arms.items())
}

contrast_rows = []
contrast_distribution_rows = []
for contrast_name, coefficients in contrast_definitions.items():
    observed = sum(
        coefficient * observed_matrix[arm]
        for arm, coefficient in coefficients.items()
    )
    null = sum(
        coefficient * null_matrix[arm]
        for arm, coefficient in coefficients.items()
    )
    bootstrap = sum(
        coefficient * bootstrap_matrix[arm]
        for arm, coefficient in coefficients.items()
    )
    for statistic_index, statistic in enumerate(STATISTICS):
        null_values = null[:, statistic_index]
        observed_value = observed[statistic_index]
        null_mean = float(np.mean(null_values))
        centered_observed = observed_value - null_mean
        centered_null = null_values - null_mean
        bootstrap_values = bootstrap[:, statistic_index]
        contrast_rows.append({
            "contrast": contrast_name,
            "statistic": statistic,
            "observed_raw_difference": observed_value,
            "null_difference_mean": null_mean,
            "observed_minus_null_expectation": centered_observed,
            "null_difference_q025": float(np.quantile(null_values, 0.025)),
            "null_difference_q975": float(np.quantile(null_values, 0.975)),
            "randomization_p_greater": (
                1 + np.sum(centered_null >= centered_observed)
            ) / (1 + len(centered_null)),
            "randomization_p_two_sided": (
                1 + np.sum(np.abs(centered_null) >= abs(centered_observed))
            ) / (1 + len(centered_null)),
            "edge_bootstrap_ci025": float(np.quantile(bootstrap_values, 0.025)),
            "edge_bootstrap_ci975": float(np.quantile(bootstrap_values, 0.975)),
            "edge_bootstrap_interpretation": (
                "descriptive sensitivity; not retraining or population uncertainty"
            ),
            "n_null_draws": len(null_values),
            "n_edge_bootstrap": len(bootstrap_values),
        })
        contrast_distribution_rows.extend({
            "contrast": contrast_name,
            "draw": draw,
            "statistic": statistic,
            "null_difference": null_values[draw],
            "edge_bootstrap_difference": bootstrap_values[draw]
            if draw < len(bootstrap_values) else np.nan,
        } for draw in range(len(null_values)))

factorial_contrasts = pd.DataFrame(contrast_rows)


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values, kind="mergesort")
    ranked = values[order]
    adjusted = ranked * len(values) / np.arange(1, len(values) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    out = np.empty_like(adjusted)
    out[order] = np.clip(adjusted, 0.0, 1.0)
    return out


factorial_contrasts["fdr_global_15_two_sided"] = bh_adjust(
    factorial_contrasts.randomization_p_two_sided
)
factorial_contrasts["fdr_within_contrast_two_sided"] = (
    factorial_contrasts.groupby("contrast", group_keys=False)[
        "randomization_p_two_sided"
    ].transform(lambda values: bh_adjust(values.to_numpy(dtype=float)))
)
factorial_contrasts.to_csv(
    OUTPUT_DIR / "factorial_paired_contrasts.csv", index=False
)
display(factorial_signature)
display(factorial_contrasts)


colors = {
    "InterGATE_raw": "#2F6FB0",
    "InterGATE_module": "#103D6E",
    "q_raw": "#E58B2A",
    "q_module": "#A94F00",
}
fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.5), constrained_layout=True)
for statistic_index, (axis, statistic) in enumerate(zip(axes, STATISTICS)):
    block = factorial_signature[factorial_signature.statistic.eq(statistic)].set_index("arm")
    arms = list(factorial_arms)
    values = block.loc[arms, "fold_over_null"].to_numpy(dtype=float)
    axis.bar(
        np.arange(len(arms)), values,
        color=[colors[arm] for arm in arms], width=0.72,
    )
    axis.axhline(1.0, color="#555555", linestyle="--", linewidth=1)
    axis.set_xticks(np.arange(len(arms)))
    axis.set_xticklabels(
        [name.replace("InterGATE_", "InterGATE\n").replace("q_", "$q$\n") for name in arms],
        rotation=0,
    )
    axis.set_ylabel("Fold over candidate-matched null")
    axis.set_title(statistic.replace("_", " "))
    for x, value in enumerate(values):
        axis.text(x, value + 0.04, f"{value:.2f}", ha="center", va="bottom", fontsize=9)
fig.suptitle(
    "Factorial control: learning × module-aware selection",
    fontsize=15,
    fontweight="bold",
)
for extension in ("pdf", "svg", "png"):
    options = {"bbox_inches": "tight"}
    if extension == "png":
        options["dpi"] = 450
    fig.savefig(FIGURE_DIR / f"review_factorial_q_control.{extension}", **options)
plt.show()


## 3. Curva de núcleo frente a $k/R$

El panel izquierdo utiliza las cinco redes existentes y varía el umbral
$k=1,\ldots,R$. El panel derecho enumera **todas** las combinaciones de $R$
semillas y calcula la intersección estricta $R/R$. Con cinco semillas es una
sensibilidad exacta al subconjunto de semillas, pero no demuestra invariancia
frente a aumentar $R$; para ello se pueden añadir más artefactos densos a
`INTERGATE_DENSE_ROOT` y definir `INTERGATE_CORE_SEEDS`.


In [ ]:
def _parse_seed_list(value, default):
    if not value:
        return tuple(map(int, default))
    return tuple(int(token.strip()) for token in value.split(",") if token.strip())


CORE_SEEDS = _parse_seed_list(
    os.environ.get("INTERGATE_CORE_SEEDS"), LOCKED_SEEDS
)
if len(set(CORE_SEEDS)) != len(CORE_SEEDS) or len(CORE_SEEDS) < 2:
    raise ValueError("INTERGATE_CORE_SEEDS requiere al menos dos semillas únicas.")

core_seed_graphs = {"raw": {}, "module": {}}
for selector in core_seed_graphs:
    for seed in CORE_SEEDS:
        if seed in seed_graphs[selector]:
            frame = seed_graphs[selector][seed]
        else:
            frame = rebuild_seed_graph(
                dense_path_for_seed(seed), seed=seed, selector=selector
            )
        core_seed_graphs[selector][seed] = set(
            frame.candidate_edge_id.astype(int)
        )

threshold_rows = []
subset_rows = []
for selector, by_seed in core_seed_graphs.items():
    counter = Counter(edge for values in by_seed.values() for edge in values)
    R = len(by_seed)
    for k in range(1, R + 1):
        ids = {edge for edge, count in counter.items() if count >= k}
        nodes = set(candidates.loc[
            candidates.candidate_edge_id.isin(ids), "source_idx"
        ].astype(int)) | set(candidates.loc[
            candidates.candidate_edge_id.isin(ids), "target_idx"
        ].astype(int))
        threshold_rows.append({
            "selector": selector,
            "R": R,
            "k": k,
            "threshold": f"{k}/{R}",
            "n_edges": len(ids),
            "n_nodes": len(nodes),
        })
    seed_values = tuple(by_seed)
    for subset_size in range(2, R + 1):
        for subset in combinations(seed_values, subset_size):
            strict = set.intersection(*(by_seed[seed] for seed in subset))
            nodes = set(candidates.loc[
                candidates.candidate_edge_id.isin(strict), "source_idx"
            ].astype(int)) | set(candidates.loc[
                candidates.candidate_edge_id.isin(strict), "target_idx"
            ].astype(int))
            subset_rows.append({
                "selector": selector,
                "subset_R": subset_size,
                "seeds": ",".join(map(str, subset)),
                "strict_edges_RofR": len(strict),
                "strict_nodes_RofR": len(nodes),
            })

core_threshold_curve = pd.DataFrame(threshold_rows)
core_subset_sensitivity = pd.DataFrame(subset_rows)
core_subset_summary = (
    core_subset_sensitivity
    .groupby(["selector", "subset_R"], as_index=False)
    .agg(
        n_subsets=("seeds", "size"),
        mean_strict_edges=("strict_edges_RofR", "mean"),
        median_strict_edges=("strict_edges_RofR", "median"),
        q025_strict_edges=("strict_edges_RofR", lambda x: np.quantile(x, 0.025)),
        q975_strict_edges=("strict_edges_RofR", lambda x: np.quantile(x, 0.975)),
        min_strict_edges=("strict_edges_RofR", "min"),
        max_strict_edges=("strict_edges_RofR", "max"),
    )
)
core_threshold_curve.to_csv(OUTPUT_DIR / "core_threshold_k_over_R.csv", index=False)
core_subset_sensitivity.to_csv(
    OUTPUT_DIR / "core_strict_subset_sensitivity.csv", index=False
)
core_subset_summary.to_csv(
    OUTPUT_DIR / "core_strict_subset_summary.csv", index=False
)
display(core_threshold_curve)
display(core_subset_summary)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)
selector_colors = {"raw": "#4C78A8", "module": "#C44E52"}
for selector, block in core_threshold_curve.groupby("selector"):
    axes[0].plot(
        block.k, block.n_edges, marker="o", linewidth=2.2,
        color=selector_colors[selector], label=selector,
    )
axes[0].set_xticks(range(1, len(CORE_SEEDS) + 1))
axes[0].set_xlabel(f"Minimum selections k (R={len(CORE_SEEDS)})")
axes[0].set_ylabel("Edges retained at ≥k/R")
axes[0].set_title("A  Threshold core curve")
axes[0].legend(frameon=False)

positions = np.arange(2, len(CORE_SEEDS) + 1)
for offset, selector in zip((-0.13, 0.13), ("raw", "module")):
    blocks = [
        core_subset_sensitivity.loc[
            core_subset_sensitivity.selector.eq(selector)
            & core_subset_sensitivity.subset_R.eq(R),
            "strict_edges_RofR",
        ].to_numpy(dtype=float)
        for R in positions
    ]
    box = axes[1].boxplot(
        blocks,
        positions=positions + offset,
        widths=0.22,
        patch_artist=True,
        manage_ticks=False,
        showfliers=False,
    )
    for patch in box["boxes"]:
        patch.set_facecolor(selector_colors[selector])
        patch.set_alpha(0.65)
    for median in box["medians"]:
        median.set_color("white")
        median.set_linewidth(1.7)
axes[1].set_xticks(positions)
axes[1].set_xlabel("Number of seeds in subset R")
axes[1].set_ylabel("Strict intersection size R/R")
axes[1].set_title("B  All seed-subset intersections")
axes[1].plot([], [], color=selector_colors["raw"], linewidth=7, alpha=0.65, label="raw")
axes[1].plot([], [], color=selector_colors["module"], linewidth=7, alpha=0.65, label="module")
axes[1].legend(frameon=False)
fig.suptitle("Sensitivity of the Basal-like core definition", fontsize=15, fontweight="bold")
for extension in ("pdf", "svg", "png"):
    options = {"bbox_inches": "tight"}
    if extension == "png":
        options["dpi"] = 450
    fig.savefig(FIGURE_DIR / f"review_core_k_over_R.{extension}", **options)
plt.show()


## 4. Auditoría de familias nulas y consistencia numérica

Variar la semilla del nulo produce pequeñas diferencias Monte Carlo legítimas,
pero una misma cifra primaria no debe cambiar entre Resumen, Resultados y
Tabla S5. Este bloque no sobrescribe resultados bloqueados: genera una tabla
canónica y documenta las cifras específicas de sensibilidad $\kappa$ y de las
ablaciones.


In [ ]:
# La familia canónica se recalcula aquí desde el núcleo modular 5/5 y el
# mismo nulo candidato usado por este notebook. Así Resumen, Resultados y
# suplemento pueden reutilizar una única extracción numérica reproducible.
canonical_strict_ids = set.intersection(*(
    set(frame.candidate_edge_id.astype(int))
    for frame in seed_graphs["module"].values()
))
locked_strict_ids = set(
    locked_consensus.loc[
        locked_consensus.selection_count.eq(len(LOCKED_SEEDS)),
        "candidate_edge_id",
    ].astype(int)
)
if canonical_strict_ids != locked_strict_ids:
    raise RuntimeError(
        "El núcleo modular 5/5 reconstruido no coincide con el lock V8."
    )
canonical_ids = np.asarray(sorted(canonical_strict_ids), dtype=np.int64)
canonical_observed = signature_vector(canonical_ids)
canonical_null = matched_null_matrix(
    canonical_ids, n_draws=N_MATCHED_DRAWS, seed=RANDOM_SEED
)
canonical_rows = []
for statistic_index, statistic in enumerate(STATISTICS):
    null_values = canonical_null[:, statistic_index]
    observed_value = canonical_observed[statistic_index]
    null_mean = float(np.mean(null_values))
    canonical_rows.append({
        "panel": "Basal-like_strict_5of5",
        "term": TARGET_CLASS,
        "statistic": statistic,
        "observed": observed_value,
        "null_mean": null_mean,
        "null_sd": float(np.std(null_values, ddof=1)),
        "null_q025": float(np.quantile(null_values, 0.025)),
        "null_q975": float(np.quantile(null_values, 0.975)),
        "fold_over_null": observed_value / null_mean,
        "empirical_p": (
            1 + np.sum(null_values >= observed_value)
        ) / (1 + len(null_values)),
        "n_draws": len(null_values),
        "null_seed": RANDOM_SEED,
        "null_definition": (
            "candidate-matched by edge type, sign and directed degree bins; "
            "selected edges excluded from each sampling pool"
        ),
        "reporting_role": "primary canonical candidate-matched family",
    })
canonical = pd.DataFrame(canonical_rows)
canonical_file = OUTPUT_DIR / "canonical_primary_strict_core_metrics.csv"
canonical.to_csv(canonical_file, index=False)

null_family_audit = canonical[[
    "statistic", "observed", "null_mean", "fold_over_null",
    "empirical_p", "n_draws",
]].copy()
null_family_audit.insert(0, "family", "primary_directional_strict")
null_family_audit.insert(1, "source_file", canonical_file.name)
null_family_audit["canonical_primary_fold"] = (
    null_family_audit.fold_over_null
)
null_family_audit["difference_from_canonical_fold"] = 0.0
null_family_audit["recommended_action"] = (
    "reuse this exact row in Abstract, primary Results and Supplementary Table S5"
)
null_family_audit.to_csv(
    OUTPUT_DIR / "numerical_null_family_audit.csv", index=False
)
display(canonical)
display(null_family_audit)

tex_hits = []
for tex_name in ("interGATE_main_revised_EN.tex", "InterGATE_supplement_revised_EN.tex"):
    tex_path = MANUSCRIPT_ROOT / tex_name
    if tex_path.is_file():
        for line_number, line in enumerate(
            tex_path.read_text(encoding="utf-8").splitlines(), start=1
        ):
            if re.search(r"(?:2\.4[5-9]|6\.(?:64|83))", line):
                tex_hits.append({
                    "file": tex_name,
                    "line": line_number,
                    "text": line.strip(),
                })
pd.DataFrame(tex_hits).to_csv(
    OUTPUT_DIR / "manuscript_numeric_occurrences_to_review.csv", index=False
)


## 5. Lectura correcta de la concordancia externa ya congelada

Un $P$ unilateral inferior a 0.05 puede coexistir con un observado dentro del
intervalo nulo central del 95%: el límite superior de ese intervalo es el
percentil 97.5, mientras que la prueba unilateral usa el percentil 95. La
interpretación debe ser “direccionalmente consistente pero marginal” para
TCGA-BRCA y GSE21653, y más fuerte para GSE1456 y el metaanálisis.

La concordancia de signo es un resultado secundario claro y merece visibilidad,
pero no debe ascenderse retrospectivamente a co-primario.


In [ ]:
external_summary_file = (
    RESULTS_ROOT / "08b_external_structural_concordance"
    / "external_concordance_matched_null_summary.csv"
)
external_summary = pd.read_csv(external_summary_file)
external_primary = external_summary.loc[
    external_summary.metric.eq("spearman_development_external")
].copy()
external_primary["position_vs_central_95_null"] = np.select(
    [
        external_primary.observed > external_primary.null_q975,
        external_primary.observed < external_primary.null_q025,
    ],
    ["above 97.5th percentile", "below 2.5th percentile"],
    default="inside central 95% null interval",
)
external_primary["recommended_wording"] = np.where(
    external_primary.scope.isin(["TCGA-BRCA", "GSE21653"]),
    "directionally consistent but marginal",
    "clearer evidence above the matched-null distribution",
)
external_primary.to_csv(
    OUTPUT_DIR / "external_concordance_interpretation.csv", index=False
)
external_sign = external_summary.loc[
    external_summary.metric.eq("sign_agreement_development_external")
].copy()
external_sign["reporting_role"] = "prominent secondary; not post-hoc co-primary"
external_sign.to_csv(
    OUTPUT_DIR / "external_sign_agreement_secondary.csv", index=False
)
display(external_primary[[
    "scope", "observed", "null_mean", "null_q975", "empirical_p_upper",
    "position_vs_central_95_null", "recommended_wording",
]])
display(external_sign[["scope", "observed", "null_mean", "empirical_p_upper", "reporting_role"]])


## 6. Nulo de etiquetas con reentrenamiento completo (49 permutaciones)

Esta prueba permuta etiquetas dentro de cada estrato `(cohorte, split)` para
conservar exactamente los recuentos por clase y la composición train/validation.
Después recalcula $q$, reentrena InterGATE y repite el soporte por eliminación
de cohorte. Cada permutación usa las mismas dos semillas y se compara con el
núcleo observado 2/2 de esas mismas semillas; nunca se compara un nulo 2/2 con
el núcleo observado 5/5.

- 10 permutaciones: $P_{min}=1/11=0.091$; solo exploratorio.
- 20 permutaciones: $P_{min}=1/21=0.0476$; mínimo técnico.
- 49 o 99: recomendación para una revisión.

El ejecutor existente completa también Phase 2. Esto es conservador y reproduce
el pipeline de principio a fin, aunque la red biológica ya queda fijada al final
de Phase 1 + soporte. Las ejecuciones son reanudables y pueden dividirse en lotes.


In [ ]:
def permute_labels_within_cohort_split(runtime, *, permutation_seed):
    y = np.asarray(runtime["y"], dtype=np.int64)
    cohort = np.asarray(runtime["cohort"]).astype(str)
    train_idx = np.asarray(runtime["train_idx"], dtype=np.int64)
    val_idx = np.asarray(runtime["val_idx"], dtype=np.int64)
    split = np.full(len(y), "", dtype=object)
    split[train_idx] = "train"
    split[val_idx] = "validation"
    if np.any(split == ""):
        raise RuntimeError("El runtime development-only contiene muestras sin split.")
    rng = np.random.default_rng(int(permutation_seed))
    permuted = y.copy()
    for cohort_name in sorted(np.unique(cohort)):
        for split_name in ("train", "validation"):
            ids = np.flatnonzero((cohort == cohort_name) & (split == split_name))
            if len(ids):
                permuted[ids] = rng.permutation(y[ids])
                if not np.array_equal(
                    np.bincount(y[ids], minlength=len(runtime["classes"])),
                    np.bincount(permuted[ids], minlength=len(runtime["classes"])),
                ):
                    raise RuntimeError("La permutación alteró los recuentos del estrato.")
    return permuted


def _label_hash(values):
    return sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def register_permuted_runtime(runtime, y_permuted):
    from intergate.ablation import register_runtime

    biology_config = DifferentialEvidenceConfig(
        min_samples_per_group=int(CFG.BIOLOGY_MIN_SAMPLES_PER_GROUP),
        min_cohorts=int(CFG.BIOLOGY_MIN_COHORTS),
        edge_batch_size=int(CFG.BIOLOGY_EDGE_BATCH_SIZE),
        meta_z_clip=float(CFG.BIOLOGY_META_Z_CLIP),
        heterogeneity_penalty=float(CFG.BIOLOGY_HETEROGENEITY_PENALTY),
    )
    evidence = compute_differential_edge_evidence(
        np.asarray(runtime["X_scaled"], dtype=np.float32),
        np.asarray(y_permuted, dtype=np.int64),
        runtime["cohort"],
        runtime["train_idx"],
        runtime["edge_index"],
        runtime["classes"],
        CFG.TRAIN_COHORTS,
        config=biology_config,
    )
    class_names = np.asarray(runtime["classes"], dtype=object)
    register_runtime(
        y=np.asarray(y_permuted, dtype=np.int64),
        y_str=class_names[np.asarray(y_permuted, dtype=np.int64)],
        biology_evidence=evidence["context_score"],
        biology_evidence_bundle=evidence,
    )
    return evidence


def restore_observed_runtime(runtime):
    from intergate.ablation import register_runtime

    register_runtime(
        y=np.asarray(runtime["y"], dtype=np.int64),
        y_str=np.asarray(runtime["y_str"]).astype(str),
        biology_evidence=runtime["biology_evidence"]["context_score"],
        biology_evidence_bundle=runtime["biology_evidence"],
    )


def strict_core_table(first, second):
    ids = set(first.candidate_edge_id.astype(int)) & set(
        second.candidate_edge_id.astype(int)
    )
    return candidate_lookup.loc[sorted(ids)].reset_index(drop=True)


def signature_fold_for_core(core, *, seed, n_draws):
    if core.empty:
        return {
            f"observed_{statistic}": np.nan for statistic in STATISTICS
        } | {f"fold_{statistic}": np.nan for statistic in STATISTICS}
    ids = core.candidate_edge_id.to_numpy(dtype=np.int64)
    observed = signature_vector(ids)
    null = matched_null_matrix(ids, n_draws=n_draws, seed=seed)
    out = {}
    for index, statistic in enumerate(STATISTICS):
        out[f"observed_{statistic}"] = observed[index]
        out[f"fold_{statistic}"] = observed[index] / np.mean(null[:, index])
    return out


def run_label_permutation_batch():
    from intergate.runtime_setup import prepare_runtime
    from intergate.transport import TransportConfig, run_transport_seed

    runtime = prepare_runtime(CFG, include_external=False)
    if runtime["external_opened"] or len(runtime["test_idx"]):
        raise RuntimeError("El nulo de etiquetas debe ejecutarse con externo cerrado.")
    artifacts_root = Path(CFG.ARTIFACTS_ROOT)
    permutation_root = OUTPUT_DIR / "label_permutation_retraining"
    permutation_root.mkdir(parents=True, exist_ok=True)
    rows = []
    completed_files = sorted(permutation_root.glob("permutation_*.json"))
    for path in completed_files:
        rows.append(json.loads(path.read_text(encoding="utf-8")))

    try:
        for permutation_id in range(PERMUTATION_START, PERMUTATION_STOP):
            result_file = permutation_root / f"permutation_{permutation_id:03d}.json"
            if result_file.is_file():
                print(f"[perm {permutation_id:03d}] ya resumida; se reutiliza")
                continue
            permutation_seed = RANDOM_SEED + 1_000_000 + permutation_id
            y_permuted = permute_labels_within_cohort_split(
                runtime, permutation_seed=permutation_seed
            )
            label_hash = _label_hash(y_permuted)
            if label_hash == _label_hash(runtime["y"]):
                raise RuntimeError("La permutación no cambió ninguna etiqueta.")
            artifact_name = f"INTERGATE_V8_LABELPERM_{permutation_id:03d}"
            contract_file = artifacts_root / artifact_name / "permutation_contract.json"
            contract = {
                "permutation_id": permutation_id,
                "permutation_seed": permutation_seed,
                "label_sha256": label_hash,
                "stratification": "cohort_by_train_validation_split",
                "model_seeds": list(PERMUTATION_MODEL_SEEDS),
                "external_opened": False,
            }
            if contract_file.is_file():
                observed_contract = json.loads(contract_file.read_text(encoding="utf-8"))
                if observed_contract != contract:
                    raise RuntimeError(f"Contrato incompatible en {contract_file}")
            else:
                contract_file.parent.mkdir(parents=True, exist_ok=True)
                contract_file.write_text(
                    json.dumps(contract, indent=2, ensure_ascii=False),
                    encoding="utf-8",
                )
            print(f"[perm {permutation_id:03d}] recalculando q y entrenando")
            register_permuted_runtime(runtime, y_permuted)
            graphs = []
            for model_seed in PERMUTATION_MODEL_SEEDS:
                run_transport_seed(
                    model_seed,
                    save_root=str(artifacts_root),
                    transport=TransportConfig.from_global(CFG),
                    resume=True,
                    force=False,
                    artifact_name=artifact_name,
                )
                dense_file = (
                    artifacts_root / artifact_name / f"seed_{model_seed}"
                    / "transport_support_dense.npz"
                )
                graphs.append(rebuild_seed_graph(
                    dense_file, seed=model_seed, selector="module"
                ))
            core = strict_core_table(graphs[0], graphs[1])
            nodes = set(core.source_idx.astype(int)) | set(core.target_idx.astype(int))
            row = {
                **contract,
                "strict_edges_2of2": int(len(core)),
                "strict_nodes_2of2": int(len(nodes)),
                **signature_fold_for_core(
                    core,
                    seed=RANDOM_SEED + 2_000_000 + permutation_id,
                    n_draws=N_PERM_MATCHED_DRAWS,
                ),
            }
            result_file.write_text(
                json.dumps(row, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            rows = [entry for entry in rows if entry["permutation_id"] != permutation_id]
            rows.append(row)
            pd.DataFrame(rows).sort_values("permutation_id").to_csv(
                permutation_root / "label_permutation_results.csv", index=False
            )
    finally:
        restore_observed_runtime(runtime)
    return pd.DataFrame(rows).sort_values("permutation_id").reset_index(drop=True)


def summarize_label_permutations(permutation_results):
    observed_graphs = [
        seed_graphs["module"][seed]
        for seed in PERMUTATION_MODEL_SEEDS
    ]
    observed_core = strict_core_table(observed_graphs[0], observed_graphs[1])
    observed_nodes = set(observed_core.source_idx.astype(int)) | set(
        observed_core.target_idx.astype(int)
    )
    observed = {
        "strict_edges_2of2": len(observed_core),
        "strict_nodes_2of2": len(observed_nodes),
        **signature_fold_for_core(
            observed_core,
            seed=RANDOM_SEED + 3_000_000,
            n_draws=N_PERM_MATCHED_DRAWS,
        ),
    }
    metrics = [
        "strict_edges_2of2",
        "strict_nodes_2of2",
        "fold_at_least_one_endpoint",
        "fold_both_endpoints",
        "fold_unique_node_fraction",
    ]
    rows = []
    for metric in metrics:
        null = pd.to_numeric(permutation_results[metric], errors="coerce").dropna().to_numpy(float)
        value = float(observed[metric])
        rows.append({
            "metric": metric,
            "observed_same_two_seeds": value,
            "permutation_mean": float(np.mean(null)) if len(null) else np.nan,
            "permutation_q025": float(np.quantile(null, 0.025)) if len(null) else np.nan,
            "permutation_q975": float(np.quantile(null, 0.975)) if len(null) else np.nan,
            "empirical_p_upper": (
                (1 + np.sum(null >= value)) / (1 + len(null))
                if len(null) else np.nan
            ),
            "completed_permutations": len(null),
            "minimum_attainable_p": 1 / (1 + len(null)) if len(null) else np.nan,
        })
    return pd.DataFrame(rows)


if RUN_LABEL_PERMUTATIONS:
    permutation_results = run_label_permutation_batch()
else:
    permutation_file = (
        OUTPUT_DIR / "label_permutation_retraining" / "label_permutation_results.csv"
    )
    permutation_results = (
        pd.read_csv(permutation_file) if permutation_file.is_file() else pd.DataFrame()
    )
    print(
        "Permutaciones apagadas. Para ejecutar un lote:\n"
        "INTERGATE_RUN_LABEL_PERMUTATIONS=1 "
        "INTERGATE_N_LABEL_PERMUTATIONS=49 "
        "INTERGATE_PERM_START=0 INTERGATE_PERM_STOP=10 "
        "jupyter nbconvert --execute --to notebook "
        "notebooks/12_controles_revision_q_permutaciones.ipynb "
        "--ExecutePreprocessor.timeout=-1"
    )

if not permutation_results.empty:
    permutation_summary = summarize_label_permutations(permutation_results)
    permutation_summary.to_csv(
        OUTPUT_DIR / "label_permutation_retraining" / "label_permutation_summary.csv",
        index=False,
    )
    display(permutation_summary)
else:
    permutation_summary = pd.DataFrame()


## 7. Manifiesto y lectura final


In [ ]:
output_files = sorted(
    path for path in OUTPUT_DIR.rglob("*")
    if path.is_file() and path.name != "reviewer_controls_manifest.json"
)
manifest_out = {
    "analysis_status": "posthoc_reviewer_motivated",
    "external_expression_or_labels_opened": False,
    "target_class": TARGET_CLASS,
    "locked_seeds": list(LOCKED_SEEDS),
    "exact_k": EXPECTED_K,
    "score_weights": {
        "differential": 1.0,
        "context": LOCKED_CONTEXT_WEIGHT,
        "support": LOCKED_SUPPORT_WEIGHT,
    },
    "factorial_cells": list(factorial_arms),
    "matched_null_draws": N_MATCHED_DRAWS,
    "edge_bootstrap_draws": N_EDGE_BOOTSTRAP,
    "edge_bootstrap_scope": (
        "descriptive edge-resampling sensitivity; not retraining uncertainty"
    ),
    "label_permutation_requested": RUN_LABEL_PERMUTATIONS,
    "label_permutation_completed": int(len(permutation_results)),
    "label_permutation_total_target": N_LABEL_PERMUTATIONS,
    "input_hashes": {
        CANDIDATE_FILE.name: _sha256_file(CANDIDATE_FILE),
        SIGNATURE_SOURCE_MANIFEST.name: _sha256_file(
            SIGNATURE_SOURCE_MANIFEST
        ),
        c2_file.name: _sha256_file(c2_file),
        dense_path_for_seed(LOCKED_SEEDS[0]).name: _sha256_file(
            dense_path_for_seed(LOCKED_SEEDS[0])
        ),
        locked_consensus_file.name: _sha256_file(locked_consensus_file),
        external_summary_file.name: _sha256_file(external_summary_file),
    },
    "outputs": {
        str(path.relative_to(OUTPUT_DIR)): _sha256_file(path)
        for path in output_files
    },
}
(OUTPUT_DIR / "reviewer_controls_manifest.json").write_text(
    json.dumps(manifest_out, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(Markdown(
    "**Lectura recomendada:** incorpore primero la celda `InterGATE_raw`, "
    "el contraste directo frente a $q$, la curva $k/R$ y la armonización "
    "numérica. Describa TCGA-BRCA y GSE21653 como marginales y mantenga la "
    "concordancia de signo como secundaria prominente. Complete las 49 "
    "permutaciones de etiquetas antes de interpretar el control de "
    "reentrenamiento completo.**"
))
print("Salidas:", OUTPUT_DIR)
